# Дз-5 по Биоинформатике
### Выполнил Корняков Санан Арсланович, 2 группа
## Подготовка среды

In [1]:
!mkdir -p data

Устанавливаем HMMER

In [5]:
%%bash
wget http://eddylab.org/software/hmmer/hmmer-3.3.2.tar.gz
tar xf hmmer-3.3.2.tar.gz
cd hmmer-3.3.2

--2025-06-14 05:32:21--  http://eddylab.org/software/hmmer/hmmer-3.3.2.tar.gz
Resolving eddylab.org (eddylab.org)... 96.126.110.11, 2600:3c03::f03c:91ff:fec8:383c
Connecting to eddylab.org (eddylab.org)|96.126.110.11|:80... connected.
HTTP request sent, awaiting response... 

Process was interrupted.


CalledProcessError: Command 'b'wget http://eddylab.org/software/hmmer/hmmer-3.3.2.tar.gz\ntar xf hmmer-3.3.2.tar.gz\ncd hmmer-3.3.2\n'' died with <Signals.SIGINT: 2>.

In [ ]:
%%bash
cd hmmer-3.3.2
./configure
make
make install

Скачиваем Pfam По ftp доступны разны версии базы. Нас интересует последняя в формате hmm

In [ ]:
!wget http://ftp.ebi.ac.uk/pub/databases/Pfam/releases/Pfam35.0/Pfam-A.hmm.gz

--2025-06-08 19:30:43--  http://ftp.ebi.ac.uk/pub/databases/Pfam/releases/Pfam35.0/Pfam-A.hmm.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 293000230 (279M) [application/x-gzip]
Saving to: ‘Pfam-A.hmm.gz’

Pfam-A.hmm.gz       100%[===================>] 279,43M  41,3MB/s    in 7,4s    

2025-06-08 19:30:51 (38,0 MB/s) - ‘Pfam-A.hmm.gz’ saved [293000230/293000230]



In [ ]:
!gunzip Pfam-A.hmm.gz

Устанавливаем ZDNABERT

In [ ]:
%pip install transformers torch biopython gdown

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from torch import nn
import transformers
from transformers import BertTokenizer, BertForTokenClassification
import numpy as np
from Bio import SeqIO
from io import StringIO, BytesIO
from tqdm import tqdm
import pickle
import scipy
from scipy import ndimage

In [4]:
model = 'HG kouzine' #@param ["HG chipseq", "HG kouzine", "MM chipseq", "MM kouzine"]

In [5]:
if model == 'HG chipseq':
    model_id = '1VAsp8I904y_J0PUhAQqpSlCn1IqfG0FB'
elif model == 'HG kouzine':
    model_id = '1dAeAt5Gu2cadwDhbc7OnenUgDLHlUvkx'
elif model == 'MM curax':
    model_id = '1W6GEgHNoitlB-xXJbLJ_jDW4BF35W1Sd'
elif model == 'MM kouzine':
    model_id = '1dXpQFmheClKXIEoqcZ7kgCwx6hzVCv3H'

In [9]:
!gdown $model_id
!gdown 10sF8Ywktd96HqAL0CwvlZZUUGj05CGk5
!gdown 16bT7HDv71aRwyh3gBUbKwign1mtyLD2d
!gdown 1EE9goZ2JRSD8UTx501q71lGCk-CK3kqG
!gdown 1gZZdtAoDnDiLQqjQfGyuwt268Pe5sXW0 
    

!mkdir 6-new-12w-0
!mv pytorch_model.bin 6-new-12w-0/
!mv config.json 6-new-12w-0/
!mv special_tokens_map.json 6-new-12w-0/
!mv tokenizer_config.json 6-new-12w-0/
!mv vocab.txt 6-new-12w-0/

Downloading...
From (original): https://drive.google.com/uc?id=1dAeAt5Gu2cadwDhbc7OnenUgDLHlUvkx
From (redirected): https://drive.google.com/uc?id=1dAeAt5Gu2cadwDhbc7OnenUgDLHlUvkx&confirm=t&uuid=84ea1d42-3c8a-4644-bfe7-f1ea67d61a34
To: /home/alexia/repos/bio/hw5/pytorch_model.bin
100%|████████████████████████████████████████| 354M/354M [00:05<00:00, 63.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=10sF8Ywktd96HqAL0CwvlZZUUGj05CGk5
To: /home/alexia/repos/bio/hw5/config.json
100%|██████████████████████████████████████████| 634/634 [00:00<00:00, 1.88MB/s]
Downloading...
From: https://drive.google.com/uc?id=16bT7HDv71aRwyh3gBUbKwign1mtyLD2d
To: /home/alexia/repos/bio/hw5/special_tokens_map.json
100%|███████████████████████████████████████████| 112/112 [00:00<00:00, 232kB/s]
Downloading...
From: https://drive.google.com/uc?id=1EE9goZ2JRSD8UTx501q71lGCk-CK3kqG
To: /home/alexia/repos/bio/hw5/tokenizer_config.json
100%|█████████████████████████████████████████| 40.0/40.0 [00:00<0

In [6]:
tokenizer = BertTokenizer.from_pretrained('6-new-12w-0/')
model = BertForTokenClassification.from_pretrained('6-new-12w-0/')
model.cuda()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(4101, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Скачиваем геном и протеом с ncbi и кладём в data

In [11]:
! wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_genomic.fna.gz
! gunzip -d GCA_964036135.1_CAEBRE_CFB2252_genomic.fna.gz

--2025-06-14 11:31:20--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_genomic.fna.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.13, 130.14.250.31, 130.14.250.7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40080125 (38M) [application/x-gzip]
Saving to: ‘GCA_964036135.1_CAEBRE_CFB2252_genomic.fna.gz’

GCA_964036135.1_CAE 100%[===================>]  38,22M  10,5MB/s    in 3,8s    

2025-06-14 11:31:25 (9,96 MB/s) - ‘GCA_964036135.1_CAEBRE_CFB2252_genomic.fna.gz’ saved [40080125/40080125]



In [12]:
! mv GCA_964036135.1_CAEBRE_CFB2252_genomic.fna data/genome.fasta

In [9]:
! wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_protein.faa.gz
! gunzip -d GCA_964036135.1_CAEBRE_CFB2252_protein.faa.gz

--2025-06-14 11:28:41--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_protein.faa.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.13, 130.14.250.31, 130.14.250.7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6141031 (5,9M) [application/x-gzip]
Saving to: ‘GCA_964036135.1_CAEBRE_CFB2252_protein.faa.gz’

GCA_964036135.1_CAE 100%[===================>]   5,86M  1,52MB/s    in 3,8s    

2025-06-14 11:28:46 (1,52 MB/s) - ‘GCA_964036135.1_CAEBRE_CFB2252_protein.faa.gz’ saved [6141031/6141031]



In [10]:
! mv GCA_964036135.1_CAEBRE_CFB2252_protein.faa data/proteome.fasta

In [36]:
! wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_genomic.gff.gz
! gunzip -d GCA_964036135.1_CAEBRE_CFB2252_genomic.gff.gz

--2025-06-14 13:13:59--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_genomic.gff.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.12, 130.14.250.13, 130.14.250.31, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8600366 (8,2M) [application/x-gzip]
Saving to: ‘GCA_964036135.1_CAEBRE_CFB2252_genomic.gff.gz.1’

GCA_964036135.1_CAE 100%[===================>]   8,20M  4,78MB/s    in 1,7s    

2025-06-14 13:14:01 (4,78 MB/s) - ‘GCA_964036135.1_CAEBRE_CFB2252_genomic.gff.gz.1’ saved [8600366/8600366]



In [37]:
! mv GCA_964036135.1_CAEBRE_CFB2252_genomic.gff data/genomic.gff

In [4]:
%pip install pybedtools

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Анализ данных

### Поиск по Pfam

Создаем нужные для hmmer'a файлы

In [14]:
!hmmpress Pfam-A.hmm

Working...    done.
Pressed and indexed 19632 HMMs (19632 names and 19632 accessions).
Models pressed into binary file:   Pfam-A.hmm.h3m
SSI index for binary model file:   Pfam-A.hmm.h3i
Profiles (MSV part) pressed into:  Pfam-A.hmm.h3f
Profiles (remainder) pressed into: Pfam-A.hmm.h3p


Индексируем файл

In [15]:
!hmmfetch --index Pfam-A.hmm.h3m 

Working...    done.
Indexed 19632 HMMs (19632 names and 19632 accessions).
SSI index written to file Pfam-A.hmm.h3m.ssi


Извлекаем информаию о нужных семествах

In [ ]:
import re
import os

def extract_pfam_ids(domain_string):
    pattern = r'\b(P[FB]\d+)\b'
    return set(re.findall(pattern, domain_string))

def build_acc_mapping(pfam_hmm_file):
    """Создает словарь для быстрого поиска полных ACC по базовым ID"""
    mapping = {}
    with open(pfam_hmm_file, 'r') as f:
        for line in f:
            if line.startswith("ACC   "):
                parts = line.split()
                if len(parts) >= 2:
                    full_acc = parts[1].strip()
                    base_id = full_acc.split('.')[0]
                    mapping[base_id] = full_acc
    return mapping

input_file = "data/domain_strings.txt"
pfam_hmm_file = "Pfam-A.hmm"
pfam_h3m_file = "Pfam-A.hmm.h3m"

print("Building ACC mapping...")
acc_mapping = build_acc_mapping(pfam_hmm_file)
print(f"Found {len(acc_mapping)} ACC mappings")

all_ids = set()

with open(input_file, 'r') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        
        print(f"Processing line {line_num}: {line}")
        pfam_ids = extract_pfam_ids(line)
        
        if len(pfam_ids) == 0:
            print(f"  No PF/PB IDs found in line {line_num}")
            continue

        missing_ids = [pid for pid in pfam_ids if pid not in acc_mapping]
            
        if missing_ids:
            print('\033[31m' + f"  Missing IDs: {', '.join(missing_ids)} - SKIPPING ENTIRE LINE" + '\033[0m')
            continue
        
        all_ids = all_ids.union(pfam_ids)

with open("selected_ids.txt", "w") as f:
    f.write('\n'.join([acc_mapping[id] for id in all_ids]))

Building ACC mapping...
Found 19632 ACC mappings
Processing line 1: Bromodomain PF00439 571-654, EPL1 PF10513 46-196, PHD_2 PF13831 228-262, PWWP PF00855 927-1042, zf-HC5HC2H_2 PF13832 269-388
Processing line 2: IQ PF00612 735-755 758-777, Myosin_TH1 PF06017 873-1059, Myosin_head PF00063 48-718
Processing line 3: PF00651;PF00096
Processing line 4: YEATS PF03366 44-126
Processing line 5: RNase_PH PF01138 35-175
Processing line 6: zf-C2H2 PF00096 220-242, zf-H2C2_2 PF13465 177-203 262-287 290-314
Processing line 7: Hist_deacetyl PF00850 23-320
Processing line 8: HMG14_17 PF01101 2-98
Processing line 9: ING PF12998 26-125, PHD PF00628 214-261
Processing line 10: UQ_con PF00179 7-144


In [24]:
! hmmfetch -f Pfam-A.hmm.h3m selected_ids.txt > Z-alpha.hmm

In [30]:
! mkdir -p tables

In [31]:
! hmmsearch --tblout tables/result_hmm.tsv Z-alpha.hmm data/proteome.fasta > /dev/null

Создаём таблицу семейств и генов

In [39]:
from collections import defaultdict
import re

protein_to_gene = {}
gene_names = {}

attr_pattern = re.compile(r'([^=;]+)=([^;]+)')

with open("data/genomic.gff", "r") as gff:
    for line in gff:
        if line.startswith('#'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 9:
            continue
        
        feature_type = parts[2]
        attributes = parts[8]
        
        attrs = {}
        for match in attr_pattern.finditer(attributes):
            key, value = match.groups()
            attrs[key] = value
        
        if feature_type == "gene":
            gene_id = attrs.get("ID")
            gene_name = attrs.get("Name", attrs.get("gene", gene_id))
            if gene_id:
                gene_names[gene_id] = gene_name
        
        elif feature_type in ["CDS", "mRNA"]:
            protein_id = attrs.get("protein_id")
            gene_id = attrs.get("Parent")
            
            if protein_id and gene_id:
                if gene_id.startswith("transcript:"):
                    gene_id = gene_names.get(gene_id, gene_id)
                protein_to_gene[protein_id] = gene_names.get(gene_id, gene_id)

family_gene_map = defaultdict(set)

with open("tables/result_hmm.tsv", "r") as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) < 4:
            continue
        
        protein_id = parts[0]
        domain_acc = parts[3]
        base_domain = domain_acc.split('.')[0]
        
        gene_name = protein_to_gene.get(protein_id, protein_id).split('-')[1]
        family_gene_map[base_domain].add(gene_name)

with open("tables/family_gene.tsv", "w") as f:
    f.write("family\tgene\n")
    for family, genes in family_gene_map.items():
        for gene in genes:
            f.write(f"{family}\t{gene}\n")

### Поиск Z-ДНК

In [41]:
import re
import os
import torch
import numpy as np
from Bio import SeqIO
from tqdm import tqdm
import scipy

def seq2kmer(seq, k):
    """Преобразует последовательность в список k-меров"""
    return [seq[x:x+k] for x in range(len(seq) - k + 1)]

def split_seq(seq, length=512, pad=16):
    """Разбивает последовательность на перекрывающиеся части"""
    res = []
    for st in range(0, len(seq), length - pad):
        end = min(st + length, len(seq))
        res.append(seq[st:end])
        if end == len(seq):
            break
    return res

def stitch_np_seq(np_seqs, pad=16):
    """Соединяет предсказания с учетом перекрытия (оптимизированная версия)"""
    if not np_seqs:
        return np.array([])
    
    total_length = 0
    for i, seq in enumerate(np_seqs):
        L = len(seq)
        if i == 0:
            total_length = L
        else:
            overlap = min(pad, total_length)
            total_length = total_length - overlap + L
    
    res = np.empty(total_length, dtype=np_seqs[0].dtype)
    current_end = 0
    
    for i, seq in enumerate(np_seqs):
        L = len(seq)
        if i == 0:
            res[0:L] = seq
            current_end = L
        else:
            overlap = min(pad, current_end)
            start_index = current_end - overlap
            res[start_index:start_index + L] = seq
            current_end = start_index + L
    
    return res

def predict_zdna_bert(sequence, min_length=6, threshold=0.9, k=6):
    kmer_seq = seq2kmer(sequence, k)
    seq_pieces = split_seq(kmer_seq)
    with torch.no_grad():
        preds = []
        for seq_piece in tqdm(seq_pieces):
            input_ids = torch.LongTensor(tokenizer.encode(' '.join(seq_piece), add_special_tokens=False))
            outputs = torch.softmax(model(input_ids.cuda().unsqueeze(0))[-1],axis = -1)[0,:,1]
            preds.append(outputs.cpu().numpy())

    labeled, max_label = scipy.ndimage.label(stitch_np_seq(preds) > threshold)
    hits = []
    min_length_kmers = max(1, min_length - k + 1)
    for label in range(1, max_label+1):
        candidate = np.where(labeled == label)[0]
        if len(candidate) >= min_length_kmers:
            start_nuc = candidate[0]
            end_nuc = candidate[-1] + k
            hits.append({
                'start': start_nuc,
                'end': end_nuc
            })
    return hits

def find_quadruplexes(sequence):
    """Находит G-квадруплексы на обеих цепях ДНК"""
    pattern = re.compile(r'(G{3,5}[ATGC]{1,7}){3,}G{3,5}', re.IGNORECASE)
    matches = []
    
    for match in pattern.finditer(sequence):
        matches.append({
            'start': match.start(),
            'end': match.end()
        })
    
    rev_sequence = reverse_complement(sequence)
    for match in pattern.finditer(rev_sequence):
        rev_start = len(sequence) - match.end()
        rev_end = len(sequence) - match.start()
        matches.append({
            'start': rev_start,
            'end': rev_end,
        })
    
    return matches

def reverse_complement(seq):
    """Возвращает обратную комплементарную последовательность"""
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N',
            'a': 't', 't': 'a', 'g': 'c', 'c': 'g'}
    return ''.join(comp[base] for base in reversed(seq))

genome = "data/genome.fasta"
output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

zbert_out_path = os.path.join(output_dir, "zdnabert_predictions.bed")
gq_out_path = os.path.join(output_dir, "quadruplexes.bed")

with open(zbert_out_path, 'w') as zbert_file, open(gq_out_path, 'w') as gq_file:
    for record in tqdm(SeqIO.parse(genome, "fasta"), desc="Processing genome"):
        seq = str(record.seq).upper()
        contig = record.id
        
        zdnabert_hits = predict_zdna_bert(seq)
        for hit in zdnabert_hits:
            zbert_file.write(f"{contig}\t{hit['start']}\t{hit['end']}\n")
        
        quadruplexes = find_quadruplexes(seq)
        for q in quadruplexes:
            gq_file.write(f"{contig}\t{q['start']}\t{q['end']}\n")

100%|██████████| 30/30 [00:00<00:00, 53.36it/s]
Processing genome: 170it [1:21:25, 28.74s/it]


In [ ]:
! ./zhunt 12 8 12 data/genome.fasta

Теперь надо преобразовать абсолютные координаты в относительные и добавить контиги

In [51]:
import bisect

def parse_gff(gff_file):
    """
    Парсит GFF файл и извлекает информацию о контигах (хромосомах)
    Возвращает список контигов в формате (contig_id, length)
    """
    contigs = []
    with open(gff_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            if fields[2] == "region":
                contig_id = fields[0]
                start = int(fields[3])
                end = int(fields[4])
                length = end - start + 1
                contigs.append((contig_id, length))
    return contigs

gff_file = "data/genomic.gff"
zscore_file = "data/genome.fna.Z-SCORE"
output_file = "results/zhunt_predictions.bed"

contigs = parse_gff(gff_file)
if not contigs:
    raise ValueError("No contigs found in GFF file")

cumulative_starts = []
contig_map = []

total_length = 0
for contig_id, length in contigs:
    contig_map.append((contig_id, length, total_length))
    cumulative_starts.append(total_length)
    total_length += length
cumulative_starts.append(total_length)

with open(zscore_file, 'r') as zfile, open(output_file, 'w') as bedfile:
    for line in zfile:
        parts = line.strip().split()
        if not parts or len(parts) < 3:
            continue
        
        start_genome = int(parts[0])
        end_genome = int(parts[1])
        
        idx = bisect.bisect_right(cumulative_starts, start_genome) - 1
        if idx < 0 or idx >= len(contig_map):
            print("Unknown contig coordinate:", start_genome)
            continue
        
        contig_id, contig_length, contig_start = contig_map[idx]

        start_contig = start_genome - contig_start
        end_contig = end_genome - contig_start
        
        if start_contig < 0:
            start_contig = 0
        if end_contig > contig_length:
            print("Start and end not in the same contig")
            end_contig = contig_length
        if start_contig >= end_contig:
            continue
        
        bedfile.write(f"{contig_id}\t{start_contig}\t{end_contig}\n")

Составляем таблицы попаданий

In [1]:
import pybedtools
import os
from collections import defaultdict

genome_fasta = "data/genome.fasta"
gff_file = "data/genomic.gff"
bed_files = {
    "quadruplexes": "results/quadruplexes.bed",
    "zhunt": "results/zhunt_predictions.bed",
    "zdnabert": "results/zdnabert_predictions.bed"
}

chrom_lengths = {}
with open(genome_fasta) as f:
    chrom = None
    for line in f:
        if line.startswith(">"):
            chrom = line[1:].split()[0]
            chrom_lengths[chrom] = 0
        else:
            chrom_lengths[chrom] += len(line.strip())

def parse_gff(gff_path):
    genes = {}
    exons = defaultdict(list)
    mrnas = {}
    
    with open(gff_path) as f:
        for line in f:
            if line.startswith("#"): 
                continue
            fields = line.strip().split("\t")
            if len(fields) < 9: 
                continue
            chrom, source, ftype, start, end, _, strand, _, attrs = fields
            start, end = int(start)-1, int(end)
            
            attr_dict = {}
            for item in attrs.split(";"):
                if "=" in item:
                    k, v = item.split("=", 1)
                    attr_dict[k] = v
            
            if ftype == "gene":
                gene_id = attr_dict.get("ID")
                if gene_id:
                    genes[gene_id] = (chrom, start, end, strand)
            elif ftype == "mRNA":
                parent = attr_dict.get("Parent")
                if parent and parent in genes:
                    mrnas[attr_dict["ID"]] = parent
            elif ftype == "exon":
                parent = attr_dict.get("Parent")
                if parent and parent in mrnas:
                    gene_id = mrnas[parent]
                    exons[gene_id].append((chrom, start, end, strand))
    return genes, exons

genes, exons_by_gene = parse_gff(gff_file)

def create_region_beds(genes, exons_by_gene, chrom_lengths):
    promoters = []
    downstreams = []
    exon_regions = []
    intron_regions = []
    
    for gene_id, (chrom, start, end, strand) in genes.items():
        chrom_len = chrom_lengths.get(chrom, 0)
        if strand == "+":
            prom_start = max(0, start - 1000)
            prom_end = start
            down_start = end
            down_end = min(chrom_len, end + 200)
        else:
            prom_start = end
            prom_end = min(chrom_len, end + 1000)
            down_start = max(0, start - 200)
            down_end = start
        
        if prom_start < prom_end:
            promoters.append((chrom, prom_start, prom_end))
        if down_start < down_end:
            downstreams.append((chrom, down_start, down_end))
    
    for gene_exons in exons_by_gene.values():
        for exon in gene_exons:
            exon_regions.append((exon[0], exon[1], exon[2]))

    for gene_id, gene_exons in exons_by_gene.items():
        if gene_id not in genes:
            continue
            
        chrom, gene_start, gene_end, strand = genes[gene_id]
        sorted_exons = sorted(gene_exons, key=lambda x: x[1])
        
        if sorted_exons[0][1] > gene_start:
            intron_regions.append((chrom, gene_start, sorted_exons[0][1]))
        
        for i in range(1, len(sorted_exons)):
            prev_end = sorted_exons[i-1][2]
            next_start = sorted_exons[i][1]
            if next_start > prev_end:
                intron_regions.append((chrom, prev_end, next_start))
        
        if sorted_exons[-1][2] < gene_end:
            intron_regions.append((chrom, sorted_exons[-1][2], gene_end))
    
    genome_regions = []
    for chrom, length in chrom_lengths.items():
        genome_regions.append((chrom, 0, length))
    
    all_features = promoters + downstreams + exon_regions + intron_regions
    all_features_bed = pybedtools.BedTool(all_features).sort().merge()
    
    genome_bed = pybedtools.BedTool(genome_regions).sort()
    intergenic_bed = genome_bed.subtract(all_features_bed)
    intergenic_regions = [(interval.chrom, interval.start, interval.end) for interval in intergenic_bed]
    
    def save_and_sort_bed(regions, filename):
        regions.sort(key=lambda x: (x[0], x[1]))
        with open(filename, "w") as f:
            for chrom, start, end in regions:
                f.write(f"{chrom}\t{start}\t{end}\n")
        return pybedtools.BedTool(filename).sort()
    
    region_beds = {}
    region_beds["promoters"] = save_and_sort_bed(promoters, "promoters.bed")
    region_beds["downstream"] = save_and_sort_bed(downstreams, "downstream.bed")
    region_beds["exons"] = save_and_sort_bed(exon_regions, "exons.bed")
    region_beds["introns"] = save_and_sort_bed(intron_regions, "introns.bed")
    region_beds["intergenic"] = save_and_sort_bed(intergenic_regions, "intergenic.bed")
    
    return region_beds

region_beds = create_region_beds(genes, exons_by_gene, chrom_lengths)

region_files = [
    ("promoters", region_beds["promoters"]),
    ("downstream", region_beds["downstream"]),
    ("exons", region_beds["exons"]),
    ("introns", region_beds["introns"])
]

results = {}
for name, bed_file in bed_files.items():
    total = sum(1 for _ in open(bed_file))
    counts = {region: 0 for region, _ in region_files}
    counts["intergenic"] = 0
    
    current_bed = pybedtools.BedTool(bed_file)
    for region, region_bed in region_files:
        in_region = current_bed.intersect(region_bed, u=True)
        count = len(in_region)
        counts[region] = count
        current_bed = current_bed.intersect(region_bed, v=True)
    
    counts["intergenic"] = len(current_bed)
    results[name] = {"total": total, "counts": counts}

table = []
regions_order = ["exons", "introns", "promoters", "downstream", "intergenic"]
region_names = {
    "exons": "Exons",
    "introns": "Introns",
    "promoters": "Promoters (1000 up from TSS)",
    "downstream": "Downstream (200 bp)",
    "intergenic": "Intergenic"
}

for region_key in regions_order:
    row = [region_names[region_key]]
    for dataset in ["quadruplexes", "zhunt", "zdnabert"]:
        count = results[dataset]["counts"][region_key]
        fraction = count / results[dataset]["total"] * 100
        row.extend([str(count), f"{fraction:.2f}%"])
    table.append(row)

header = ["Region", "Quadruplexes Count", "Quadruplexes Fraction", 
          "Zhun Count", "Zhun Fraction", 
          "ZDNABERT Count", "ZDNABERT Fraction"]
print("\t".join(header))
for row in table:
    print("\t".join(row))

Region	Quadruplexes Count	Quadruplexes Fraction	Zhun Count	Zhun Fraction	ZDNABERT Count	ZDNABERT Fraction
Exons	289	5.67%	16472	12.57%	832	17.21%
Introns	1531	30.06%	31694	24.18%	1163	24.06%
Promoters (1000 up from TSS)	927	18.20%	28131	21.47%	995	20.58%
Downstream (200 bp)	125	2.45%	2887	2.20%	107	2.21%
Intergenic	2221	43.61%	51867	39.58%	1737	35.93%


In [28]:
import pandas as pd

data = []
for region_key in regions_order:
    row = {
        "Region": region_names[region_key]
    }
    for dataset in ["quadruplexes", "zhunt", "zdnabert"]:
        count = results[dataset]["counts"][region_key]
        total = results[dataset]["total"]
        fraction = (count / total * 100) if total > 0 else 0
        row[f"{dataset}_count"] = count
        row[f"{dataset}_fraction"] = f"{fraction:.2f}%"
    data.append(row)

df_table1 = pd.DataFrame(data)
df_table1.columns = [
    "Region", 
    "Quadruplexes Count", "Quadruplexes Fraction", 
    "Zhun Count", "Zhun Fraction", 
    "ZDNABERT Count", "ZDNABERT Fraction"
]

def format_table(df, title):
    """Функция для красивого форматирования таблиц"""
    return df.style \
        .set_properties(**{'text-align': 'center'}) \
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '16px'), ('font-weight', 'bold')]}
        ]) \
        .format({
            col: "{:d}" for col in df.columns if "Count" in col or "Regions" in col
        }) \
        .hide(axis="index") \
        .set_caption(title)

styled_df = format_table(df_table1, "Distribution of Genomic Features")
display(styled_df)

Region,Quadruplexes Count,Quadruplexes Fraction,Zhun Count,Zhun Fraction,ZDNABERT Count,ZDNABERT Fraction
Exons,289,5.67%,16472,12.57%,832,17.21%
Introns,1531,30.06%,31694,24.18%,1163,24.06%
Promoters (1000 up from TSS),927,18.20%,28131,21.47%,995,20.58%
Downstream (200 bp),125,2.45%,2887,2.20%,107,2.21%
Intergenic,2221,43.61%,51867,39.58%,1737,35.93%


In [29]:
region_results = {}

for region_key in regions_order:
    # Загружаем участки региона
    region_bed = region_beds[region_key]
    total_regions = len(region_bed)
    
    # Словарь для хранения результатов по этому региону
    region_stats = {"total_regions": total_regions}
    
    # Для каждого типа предсказаний
    for dataset in ["quadruplexes", "zhunt", "zdnabert"]:
        # Загружаем предсказания
        pred_bed = pybedtools.BedTool(bed_files[dataset]).sort()
        
        # Находим участки региона, которые пересекаются хотя бы с одним предсказанием
        intersected = region_bed.intersect(pred_bed, u=True)
        count = len(intersected)
        
        # Сохраняем результаты
        region_stats[f"{dataset}_count"] = count
        region_stats[f"{dataset}_fraction"] = (count / total_regions * 100) if total_regions > 0 else 0
    
    region_results[region_key] = region_stats

# Создаем структуру данных для второй таблицы
data_table2 = []
for region_key in regions_order:
    row = {"Region": region_names[region_key]}
    stats = region_results[region_key]
    
    for dataset in ["quadruplexes", "zhunt", "zdnabert"]:
        count = stats[f"{dataset}_count"]
        fraction = stats[f"{dataset}_fraction"]
        row[f"{dataset}_count"] = count
        row[f"{dataset}_fraction"] = f"{fraction:.2f}%"
    
    data_table2.append(row)

# Создаем DataFrame для таблицы 2
df_table2 = pd.DataFrame(data_table2)
df_table2.columns = [
    "Region", 
    "Regions with Quadruplex", "Fraction with Quadruplex", 
    "Regions with Zhun", "Fraction with Zhun", 
    "Regions with ZDNABERT", "Fraction with ZDNABERT"
]

styled_table1 = format_table(df_table1, "Distribution of Structures in Genomic Regions")
styled_table2 = format_table(df_table2, "Genomic Regions Containing Structures")

In [30]:
display(styled_table1)
display(styled_table2)

Region,Quadruplexes Count,Quadruplexes Fraction,Zhun Count,Zhun Fraction,ZDNABERT Count,ZDNABERT Fraction
Exons,289,5.67%,16472,12.57%,832,17.21%
Introns,1531,30.06%,31694,24.18%,1163,24.06%
Promoters (1000 up from TSS),927,18.20%,28131,21.47%,995,20.58%
Downstream (200 bp),125,2.45%,2887,2.20%,107,2.21%
Intergenic,2221,43.61%,51867,39.58%,1737,35.93%


Region,Regions with Quadruplex,Fraction with Quadruplex,Regions with Zhun,Fraction with Zhun,Regions with ZDNABERT,Fraction with ZDNABERT
Exons,342,0.24%,2955,2.04%,724,0.50%
Introns,1175,0.99%,3587,3.01%,766,0.64%
Promoters (1000 up from TSS),838,3.27%,3560,13.90%,751,2.93%
Downstream (200 bp),161,0.63%,600,2.34%,106,0.41%
Intergenic,1209,11.11%,3086,28.37%,862,7.92%


Теперь сравним таблицы с фоном и распределением у человека

In [32]:
total_genome_length = sum(chrom_lengths.values())

region_lengths = {}
for region_key, region_bed in region_beds.items():
    region_length = 0
    for interval in region_bed:
        region_length += interval.end - interval.start
    region_lengths[region_key] = region_length

background_fractions = {}
for region_key in regions_order:
    fraction = region_lengths[region_key] / total_genome_length * 100
    background_fractions[region_key] = fraction

for region_key in regions_order:
    region_name = region_names[region_key]
    idx = df_table1[df_table1['Region'] == region_name].index[0]
    
    for dataset in ["Quadruplexes", "Zhun", "ZDNABERT"]:
        observed_fraction = float(df_table1.loc[idx, f"{dataset} Fraction"].replace('%', ''))
        expected_fraction = background_fractions[region_key]
        enrichment = observed_fraction / expected_fraction if expected_fraction > 0 else 0
        df_table1.loc[idx, f"{dataset} Enrichment"] = f"{enrichment:.2f}"

new_columns = ["Region"]
for dataset in ["Quadruplexes", "Zhun", "ZDNABERT"]:
    new_columns.extend([
        f"{dataset} Count", 
        f"{dataset} Fraction", 
        f"{dataset} Enrichment"
    ])

df_table1 = df_table1[new_columns]

In [34]:
styled_table1 = format_table(df_table1, "Distribution of Structures in Genomic Regions")
display(styled_table1)

Region,Quadruplexes Count,Quadruplexes Fraction,Quadruplexes Enrichment,Zhun Count,Zhun Fraction,Zhun Enrichment,ZDNABERT Count,ZDNABERT Fraction,ZDNABERT Enrichment
Exons,289,5.67%,0.21,16472,12.57%,0.46,832,17.21%,0.63
Introns,1531,30.06%,1.30,31694,24.18%,1.05,1163,24.06%,1.04
Promoters (1000 up from TSS),927,18.20%,0.90,28131,21.47%,1.06,995,20.58%,1.02
Downstream (200 bp),125,2.45%,0.61,2887,2.20%,0.54,107,2.21%,0.55
Intergenic,2221,43.61%,1.25,51867,39.58%,1.14,1737,35.93%,1.03


#### Гены квадруплексов и Z-ДНК

In [4]:
promoters_bed = region_beds["promoters"]

quad_genes = set()
quad_bed = pybedtools.BedTool("results/quadruplexes.bed")

quad_in_promoters = promoters_bed.intersect(quad_bed, wa=True)

for interval in quad_in_promoters:
    chrom = interval.chrom
    start = interval.start
    end = interval.end
    
    for gene_id, (g_chrom, g_start, g_end, strand) in genes.items():
        if strand == "+" and chrom == g_chrom and end == g_start:
            quad_genes.add(gene_id)
            break
        elif strand == "-" and chrom == g_chrom and start == g_end:
            quad_genes.add(gene_id)
            break

z_genes = set()
zhunt_bed = pybedtools.BedTool("results/zhunt_predictions.bed")
zdnabert_bed = pybedtools.BedTool("results/zdnabert_predictions.bed")

z_bed = zhunt_bed.cat(zdnabert_bed, postmerge=False).sort().merge()

z_in_promoters = promoters_bed.intersect(z_bed, wa=True)

for interval in z_in_promoters:
    chrom = interval.chrom
    start = interval.start
    end = interval.end
    
    for gene_id, (g_chrom, g_start, g_end, strand) in genes.items():
        if strand == "+" and chrom == g_chrom and end == g_start:
            z_genes.add(gene_id)
            break
        elif strand == "-" and chrom == g_chrom and start == g_end:
            z_genes.add(gene_id)
            break

def save_genes(gene_ids, filename):
    with open(filename, "w") as f:
        for gene_id in gene_ids:
            gene_name = gene_id.split('-')[1]
            f.write(f"{gene_name}\n")

save_genes(quad_genes, "results/genes_with_quadruplex_in_promoter.txt")
save_genes(z_genes, "results/genes_with_zDNA_in_promoter.txt")